# Final session: finish the DNS to $t=30$, then build Section 10

The run stands at $t = 22.8$ after 17.8 turnovers on the A100 at 1.62 s/step.
This notebook does the last **9,000 steps (~4 h)** and then produces everything
Section 10 of the paper needs, from the same data, in four cells.

**What the last four hours actually buy.** Statistics converge as $1/\sqrt{T}$, so
going from 22.8 to 30 turnovers tightens the error on $u_\tau$ from 0.67 % to
0.59 % — a factor of 1.15. The case for running it is that $t=30$ was the plan
and the marginal cost is small, not that the answer will change. **If the session
dies early you have lost nothing**: everything below works on whatever has
accumulated, and the run resumes from the newest checkpoint next time.

**Order:** 1 (GPU) → 2 (code) → 3 (Drive) → **4 (the run)** → 5, 6, 7, 8.
Cells 5–8 read from Drive and can be run at any time, including while the run is
going or on a later day.


In [ ]:
#@title 1. GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import subprocess
name = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                      capture_output=True, text=True).stdout.strip()
print('GPU:', name)
if 'A100' not in name:
    print('\n!! Not an A100: fp64 here is 1/32-1/64 of fp32 and the run will be ~20x slower.')


In [ ]:
#@title 2. Code (branch main)
import os
if os.path.isdir('/content/lssem/.git'):
    !cd /content/lssem && git fetch -q origin main && git reset -q --hard origin/main
else:
    !git clone -q --branch main https://github.com/chandc/Python_SEM.git /content/lssem
%cd /content/lssem
!git log --oneline -1
!pip install -q numba scipy matplotlib ninja


In [ ]:
#@title 3. Drive, and where the run stands
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/lssem_dns'   #@param {type:"string"}
OUT   = '/content/run02'
import glob, numpy as np, os
ck = sorted(glob.glob(f'{DRIVE}/checkpoint_*.npz'))
if ck:
    with np.load(ck[-1]) as z:
        print(f'newest checkpoint: {os.path.basename(ck[-1])}  step {int(z["step"])}  t = {float(z["t"]):.3f}')
    print(f'{(30-float(np.load(ck[-1])["t"]))/8e-4:,.0f} steps to t = 30 '
          f'= {(30-float(np.load(ck[-1])["t"]))/8e-4*1.62/3600:.1f} h at 1.62 s/step')
else:
    print('no checkpoint on Drive -- check DRIVE above')
print(f'\nstats snapshots: {len(glob.glob(f"{DRIVE}/stats_*.npz"))}')
!tail -3 {DRIVE}/run.log


In [ ]:
#@title 4. THE FINAL RUN — to t = 30
HOURS    = 5.0    #@param {type:"number"}
TARGET_T = 30.0   #@param {type:"number"}
#@markdown Budget a little above the ~4 h needed, so the run ends because it
#@markdown REACHED t = 30 rather than because the clock ran out.  It stops itself
#@markdown either way, cleanly, with a checkpoint on Drive.
cmd = (f'python -u colab/run_channel_dns.py --hours {HOURS} --target-t {TARGET_T} '
       f'--dt 8e-4 --every 100 --out {OUT} --drive {DRIVE} '
       f'--backend torch --precond vsbatch --weighting legacy --share 1 '
       f'--pc 1 --coarse-fp32 1')
print(cmd, flush=True)
!{cmd}


In [ ]:
#@title 5. Stationarity over the whole record
#@markdown Trend and two-halves tests with error bars from the EFFECTIVE sample
#@markdown count.  Expect ~29 independent samples at t = 30 and no significant
#@markdown drift; the mean should sit within a standard error of the prescribed 1.
FROM_T = 5.2  #@param {type:"number"}
import glob
snap = sorted(glob.glob(f'{DRIVE}/stats_*.npz'))[-1]
cmd = f'python scratch/stationarity.py {snap} --from-t {FROM_T} --out {DRIVE}/stationarity.png'
print(cmd)
!{cmd}
from IPython.display import Image, display
display(Image(f'{DRIVE}/stationarity.png'))



In [ ]:
#@title 6. SECTION 10 — profiles against five databases and the fractional-step twin
#@markdown The deliverable.  Differences the two snapshots bounding the averaging
#@markdown window (exact, because the accumulators are running sums), folds onto the
#@markdown half channel, and compares with MKM 1999, Lee & Moser 2015,
#@markdown Vreman & Kuerten 2014, Torroja and AKM 2001 — plus our own fractional-step
#@markdown run on the SAME box and mesh, which is the comparison that isolates the
#@markdown scheme from the domain.
#@markdown
#@markdown Peaks are scored below $y^+=60$ only: the minimal box reproduces the
#@markdown near-wall cycle, not the outer layer, and the figure shades the rest.
import glob, numpy as np, os
snaps = sorted(glob.glob(f'{DRIVE}/stats_*.npz'))
def t_of(f):
    with np.load(f) as z: return float(z['t'])
A = min(snaps, key=lambda f: abs(t_of(f) - FROM_T))
B = snaps[-1]
print(f'window: {os.path.basename(A)} (t={t_of(A):.2f}) -> {os.path.basename(B)} (t={t_of(B):.2f})\n')
cmd = f'python colab/section10.py {A} {B} --out {DRIVE} --label "FOSLS run02"'
!{cmd}
from IPython.display import Image, display
display(Image(f'{DRIVE}/section10_profiles.png'))


In [ ]:
#@title 7. Was the time step set by physics or by the scheme?
#@markdown Third field, ~25 turnovers apart from the first two.  Section 10 claims
#@markdown the step is physics-limited; this is the evidence, on the final state.
import glob
ck = sorted(glob.glob(f'{OUT}/checkpoint_*.npz')) or sorted(glob.glob(f'{DRIVE}/checkpoint_*.npz'))
!python scratch/dns_timescales.py {ck[-1]} --dt 8e-4


In [ ]:
#@title 8. Collect what goes into the paper
#@markdown Everything Section 10 needs, in one place on Drive.
!ls -la {DRIVE}/section10_profiles.png {DRIVE}/section10_table.md {DRIVE}/stationarity.png 2>/dev/null
print('\n--- section10_table.md (paste-ready) ---')
!cat {DRIVE}/section10_table.md
print('\n--- final state ---')
!tail -2 {DRIVE}/run.log


## What each output is for

| file on Drive | goes into |
|---|---|
| `section10_profiles.png` | Figure 7 — mean profile, fluctuations, shear stress, total-stress balance |
| `section10_table.md` | the Section 10 comparison table, already formatted |
| `stationarity.png` | the evidence that the averaging window is stationary |
| `run.log`, `diag.npz` | the run record; `stats_*.npz` are the windows |

**Reading the comparison.** The five databases disagree with each other by
0.5–2 % on these quantities, so that spread is the floor: agreement inside it is
as good as the reference data allows. The fractional-step column is the sharper
test, because it shares the box, the mesh and the time step — differences there
are the scheme alone.

**Expect one honest discrepancy.** The centreline $U^+$ runs above the
full-channel databases, as it did at $t=22.8$. Part is the minimal box, which
does not reproduce the outer layer; part is ours, since the fractional-step twin
on the same box lands closer. It belongs in the paper stated plainly.
